### MinIO → PostgreSQL Metadata Sync

Reads files from MinIO bucket and stores their metadata into PostgreSQL.

**PostgreSQL Tables:**
```
customers          → one row per customer
customer_files     → one row per file (with type, size, path, timestamps)
sync_log           → tracks every sync run (new/updated files detected)
```

**Auto-update strategy:**
- Run `sync_metadata()` on a schedule (or manually) - it detects new customers and new files automatically
- Only inserts rows that don't already exist - safe to re-run anytime

#### 1. Install & Import

In [1]:
!pip install minio psycopg2-binary python-dotenv --quiet


[notice] A new release of pip is available: 23.2.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
import os
import psycopg2
from minio import Minio
from datetime import datetime, timezone
from dotenv import load_dotenv

load_dotenv()


True

#### 2. Connect to MinIO & PostgreSQL

In [6]:
def get_minio():
    return Minio(
        "localhost:9000",
        access_key=os.getenv("MINIO_ROOT_USER"),
        secret_key=os.getenv("MINIO_ROOT_PASSWORD"),
        secure=False
    )

def get_pg():
    return psycopg2.connect(
        host="localhost",
        port=5432,
        dbname=os.getenv("POSTGRES_DB"),
        user=os.getenv("POSTGRES_USER"),
        password=os.getenv("POSTGRES_PASSWORD")
    )

minio  = get_minio()
pg     = get_pg()
print("✅ Connected to MinIO and PostgreSQL")

✅ Connected to MinIO and PostgreSQL


#### 3. Create Tables (run once)

In [7]:
CREATE_TABLES = """

CREATE TABLE IF NOT EXISTS customers (
    customer_id   TEXT PRIMARY KEY,
    first_seen    TIMESTAMPTZ DEFAULT NOW()
);

CREATE TABLE IF NOT EXISTS customer_files (
    id            SERIAL PRIMARY KEY,
    customer_id   TEXT REFERENCES customers(customer_id),
    file_name     TEXT,
    data_type     TEXT,          -- blood-reports | dicom | genomics | wearables
    object_path   TEXT UNIQUE,   -- full MinIO object key
    size_bytes    BIGINT,
    file_ext      TEXT,
    last_modified TIMESTAMPTZ,
    indexed_at    TIMESTAMPTZ DEFAULT NOW()
);

CREATE TABLE IF NOT EXISTS sync_log (
    id            SERIAL PRIMARY KEY,
    run_at        TIMESTAMPTZ DEFAULT NOW(),
    new_customers INT,
    new_files     INT
);

"""

def create_tables(pg):
    with pg.cursor() as cur:
        cur.execute(CREATE_TABLES)
    pg.commit()
    print("✅ Tables ready")

create_tables(pg)

✅ Tables ready


#### 4. Parse Object Path → Metadata

Extracts `customer_id`, `data_type`, `file_name` from a MinIO object key like:
`customers/CUST_amara_patel_03630F04/blood-reports/BloodReport.pdf`

In [8]:
def parse_object(obj):
    """Extract structured metadata from a MinIO object."""
    parts = obj.object_name.split("/")
    # expected: ['customers', 'CUST_xxx', 'data-type', 'filename']
    if len(parts) < 4:
        return None

    _, customer_id, data_type, file_name = parts[0], parts[1], parts[2], parts[3]
    return {
        "customer_id":   customer_id,
        "file_name":     file_name,
        "data_type":     data_type,
        "object_path":   obj.object_name,
        "size_bytes":    obj.size,
        "file_ext":      file_name.rsplit(".", 1)[-1].lower() if "." in file_name else "",
        "last_modified": obj.last_modified
    }

#### 5. Upsert Customer & File Rows

In [10]:
def upsert_customer(cur, customer_id):
    """Insert customer if not already present. Returns True if new."""
    cur.execute("""
        INSERT INTO customers (customer_id)
        VALUES (%s)
        ON CONFLICT (customer_id) DO NOTHING
    """, (customer_id,))
    return cur.rowcount == 1  # 1 = new row inserted


def upsert_file(cur, meta):
    """Insert file metadata if not already present. Returns True if new."""
    cur.execute("""
        INSERT INTO customer_files
            (customer_id, file_name, data_type, object_path, size_bytes, file_ext, last_modified)
        VALUES
            (%(customer_id)s, %(file_name)s, %(data_type)s, %(object_path)s,
             %(size_bytes)s, %(file_ext)s, %(last_modified)s)
        ON CONFLICT (object_path) DO NOTHING
    """, meta)
    return cur.rowcount == 1

#### 6. Core Sync Function

Scans the entire MinIO bucket and syncs everything into PostgreSQL.
- ✅ Safe to re-run anytime — skips existing records
- ✅ Automatically picks up new customers and new files
- ✅ Logs every run to `sync_log`

In [11]:
BUCKET = "health-data"

def sync_metadata(minio, pg):
    """Scan MinIO bucket and sync file metadata to PostgreSQL."""
    new_customers, new_files = 0, 0

    objects = minio.list_objects(BUCKET, prefix="customers/", recursive=True)

    with pg.cursor() as cur:
        for obj in objects:
            meta = parse_object(obj)
            if not meta:
                continue

            if upsert_customer(cur, meta["customer_id"]):
                new_customers += 1
                print(f"  👤 New customer: {meta['customer_id']}")

            if upsert_file(cur, meta):
                new_files += 1
                print(f"  📄 New file: {meta['object_path']}")

        # log this sync run
        cur.execute(
            "INSERT INTO sync_log (new_customers, new_files) VALUES (%s, %s)",
            (new_customers, new_files)
        )

    pg.commit()
    print(f"\n✅ Sync done — {new_customers} new customers, {new_files} new files")


# ▶️ Run the sync
sync_metadata(minio, pg)

  👤 New customer: CUST_amara_patel_03630F04
  📄 New file: customers/CUST_amara_patel_03630F04/blood-reports/BloodReport_CUST_ama_20260511.pdf
  📄 New file: customers/CUST_amara_patel_03630F04/dicom/CT_02_CUST_ama.dcm
  📄 New file: customers/CUST_amara_patel_03630F04/dicom/XR_01_CUST_ama.dcm
  📄 New file: customers/CUST_amara_patel_03630F04/wearables/Wearable_CUST_ama_20260511.csv
  👤 New customer: CUST_carlos_rivera_5A81755B
  📄 New file: customers/CUST_carlos_rivera_5A81755B/blood-reports/BloodReport_CUST_car_20260511.pdf
  📄 New file: customers/CUST_carlos_rivera_5A81755B/dicom/MR_01_CUST_car.dcm
  📄 New file: customers/CUST_carlos_rivera_5A81755B/dicom/XR_02_CUST_car.dcm
  📄 New file: customers/CUST_carlos_rivera_5A81755B/wearables/Wearable_CUST_car_20260511.csv
  👤 New customer: CUST_fatima_alsayed_8594AA83
  📄 New file: customers/CUST_fatima_alsayed_8594AA83/blood-reports/BloodReport_CUST_fat_20260511.pdf
  📄 New file: customers/CUST_fatima_alsayed_8594AA83/dicom/CT_02_CUST_fat.dc

---
#### 7. Auto-Sync on a Schedule (Optional)

Use this to automatically re-run `sync_metadata()` every N minutes inside the notebook.

> 💡 For production, prefer a cron job or APScheduler running in a separate process.

In [ ]:
!pip install apscheduler --quiet

In [ ]:
from apscheduler.schedulers.background import BackgroundScheduler

def scheduled_sync():
    print(f"\n⏰ Scheduled sync at {datetime.now(timezone.utc).strftime('%H:%M:%S UTC')}")
    # Reconnect each time so connections don't go stale
    pg_conn = get_pg()
    sync_metadata(get_minio(), pg_conn)
    pg_conn.close()

scheduler = BackgroundScheduler()
scheduler.add_job(scheduled_sync, "interval", minutes=10)  # ✏️ change interval here
scheduler.start()

print("⏰ Scheduler started — syncing every 10 minutes")
print("   Call scheduler.shutdown() to stop")

In [ ]:
# ⛔ Run this cell to stop the scheduler when done
scheduler.shutdown()
print("Scheduler stopped.")

---
#### 8. Quick Queries — Verify the Data

In [ ]:
def run_query(pg, sql, label="Result"):
    """Run a SELECT query and print the results."""
    with pg.cursor() as cur:
        cur.execute(sql)
        rows = cur.fetchall()
        cols = [d[0] for d in cur.description]
    print(f"\n📊 {label}")
    print("  " + " | ".join(cols))
    print("  " + "-" * 60)
    for row in rows:
        print("  " + " | ".join(str(v) for v in row))


# All customers
run_query(pg, "SELECT * FROM customers ORDER BY first_seen", "All Customers")

# File counts per customer per type
run_query(pg, """
    SELECT customer_id, data_type, COUNT(*) AS files, SUM(size_bytes) AS total_bytes
    FROM customer_files
    GROUP BY customer_id, data_type
    ORDER BY customer_id, data_type
""", "Files per Customer per Type")

# Sync history
run_query(pg, "SELECT * FROM sync_log ORDER BY run_at DESC LIMIT 5", "Recent Sync Runs")